# 01 — Report world model (Stage C)

Parse training reports into symbolic evidence states. Inspect stratified matches; save aggregate QA (matched phrase categories only — no protected report text in public logs).

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.constants import REPORT_COL, STUDY_ID_COL
from src.data.metadata import load_csv_optional, resolve_paths, target_columns_from_sample
from src.symbolic.report_parser import ReportParser
from src.utils import load_config

cfg = load_config(ROOT / "configs" / "submission_001.yaml")
paths = resolve_paths(explicit_root=cfg.get("paths", {}).get("competition_root"))
sample = load_csv_optional(paths.sample_submission)
targets = target_columns_from_sample(sample)
train = load_csv_optional(paths.train_csv)
assert train is not None and REPORT_COL in train.columns, "train.csv with Report column required"
parser = ReportParser(targets=targets, parser_version=cfg.get("parser_version", "report_parser_v1"))

In [ ]:
rows = []
cat_counts = Counter()
state_counts = Counter()
for _, row in train.iterrows():
    sid = str(row[STUDY_ID_COL])
    text = str(row[REPORT_COL]) if pd.notna(row[REPORT_COL]) else ""
    for rec in parser.parse_report(text):
        rows.append({
            STUDY_ID_COL: sid,
            "finding": rec.finding,
            "state": rec.state.value,
            "confidence": rec.confidence,
            "match_category": rec.match_category,
            "rule_id": rec.rule_id,
            "contradiction": rec.contradiction,
            "parser_version": rec.parser_version,
        })
        cat_counts[rec.match_category] += 1
        state_counts[(rec.finding, rec.state.value)] += 1

evidence_df = pd.DataFrame(rows)
out = ROOT / "artifacts" / "report_evidence.parquet"
out.parent.mkdir(parents=True, exist_ok=True)
evidence_df.to_csv(ROOT / "artifacts" / "report_evidence.csv", index=False)
print("Evidence rows:", len(evidence_df))
print("Match categories:", dict(cat_counts))
display(evidence_df.head(20))

In [ ]:
# Stratified QA sample of match categories (no raw report text)
qa = (
    evidence_df[evidence_df["match_category"] != "unmentioned"]
    .groupby("match_category", group_keys=False)
    .head(5)
)
display(qa[["finding", "state", "confidence", "match_category", "rule_id", "contradiction"]])